In [ ]:
import cv2
import numpy as np
import webbrowser
import time

# HSV color ranges
COLOR_RANGES = {
    "blue":  [(np.array([100, 120, 70]), np.array([140, 255, 255]))],
    "green": [(np.array([36,  100, 70]), np.array([86,  255, 255]))],
    "red":   [(np.array([0,   120, 70]), np.array([10,  255, 255])),
              (np.array([160, 120, 70]), np.array([179, 255, 255]))],
}

WEBSITES = {
    "blue":  "https://www.facebook.com",
    "red":   "https://www.youtube.com",
    "green": "https://web.whatsapp.com/",
}

MIN_AREA     = 8000   # minimum card size in pixels
HOLD_SECONDS = 1.5    # hold card for this long to trigger
COOLDOWN     = 5.0    # seconds before re-triggering same color


def detect_color(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    hsv = cv2.GaussianBlur(hsv, (7, 7), 0)
    best, best_area = None, 0
    for color, ranges in COLOR_RANGES.items():
        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
        for lo, hi in ranges:
            mask |= cv2.inRange(hsv, lo, hi)
        kernel = np.ones((5, 5), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area > MIN_AREA and area > best_area:
                best_area = area
                best = color
    return best


cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("ERROR: Could not open webcam. Check that it is connected and not in use.")
    exit(1)

hold_start      = None
holding_color   = None
last_launched   = {}   # color -> timestamp of last launch (per-color cooldown)

print("Show a colored card to the camera:")
print("  BLUE  -> Facebook")
print("  RED   -> YouTube")
print("  GREEN -> Whatsapp")
print("Press Q to quit\n")

while True:
    ret, frame = cap.read()
    if not ret:
        print("ERROR: Failed to read frame from webcam.")
        break

    frame    = cv2.flip(frame, 1)
    detected = detect_color(frame)
    now      = time.time()

    if detected:
        # Reset timer if a different color appears
        if detected != holding_color:
            holding_color = detected
            hold_start    = now
        else:
            elapsed = now - hold_start
            if elapsed >= HOLD_SECONDS:
                # Check per-color cooldown so each color tracks independently
                last_time = last_launched.get(detected, 0)
                if now - last_time > COOLDOWN:
                    url = WEBSITES[detected]
                    print(f"Opening {detected.upper()} -> {url}")
                    webbrowser.open(url)
                    last_launched[detected] = now
                # FIX: reset hold_start AND holding_color so the bar resets
                # and we don't fire again immediately on the next frame
                holding_color = None
                hold_start    = None
    else:
        holding_color = None
        hold_start    = None

    # ── Overlay ──────────────────────────────────────────────────────────────
    h, w = frame.shape[:2]

    # Status line
    if detected:
        status_text  = f"Detected: {detected.upper()}"
        status_color = (0, 220, 0)
    else:
        status_text  = "No color detected"
        status_color = (0, 0, 220)

    cv2.putText(frame, status_text, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.85, status_color, 2)

    # Legend at the bottom
    cv2.putText(frame, "BLUE=Facebook   RED=YouTube   GREEN=Whatsapp",
                (10, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

    # Hold progress bar (only while actively holding)
    if detected and hold_start is not None:
        progress = min((now - hold_start) / HOLD_SECONDS, 1.0)
        bar_w    = int(w * 0.6)
        bar_x    = (w - bar_w) // 2
        bar_y1, bar_y2 = h - 50, h - 32

        # Background track
        cv2.rectangle(frame, (bar_x, bar_y1), (bar_x + bar_w, bar_y2),
                      (60, 60, 60), -1)
        # Filled portion
        cv2.rectangle(frame, (bar_x, bar_y1),
                      (bar_x + int(bar_w * progress), bar_y2),
                      (0, 200, 100), -1)
        # Label above bar
        pct_label = f"Hold: {int(progress * 100)}%"
        cv2.putText(frame, pct_label, (bar_x, bar_y1 - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

    cv2.imshow("Color Launcher", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

Show a colored card to the camera:
  BLUE  -> Facebook
  RED   -> YouTube
  GREEN -> Whatsapp
Press Q to quit

Opening RED -> https://www.youtube.com
Opening BLUE -> https://www.facebook.com
Opening GREEN -> https://web.whatsapp.com/
